In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# LLM 활용 프레임 분석 시스템 - EXAONE 4.0.1 최적화 버전
# ═══════════════════════════════════════════════════════════════════════
import os
import pandas as pd
import openai
from tqdm import tqdm
import re
import json
import time
import logging

# ═══════════════════════════════════════════════════════════════════════
# 1. 로깅 설정
# ═══════════════════════════════════════════════════════════════════════
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("ai_risk_frame_analysis.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# ═══════════════════════════════════════════════════════════════════════
# 2. 프레임 정의
# ═══════════════════════════════════════════════════════════════════════
ALLOWED_FRAMES = {
    "인간 목표 또는 가치와 상충하는 자체적 목표를 추구하는 AI": {
        "description": "개발자나 사용자가 의도한 목표와 다른 자체적인 목표를 추구하거나, 인간이 예상하지 못한 방식으로 행동하는 위험",
    },
    "위험한 기능을 가진 AI": {
        "description": "AI 시스템이 인간에게 비합리적이거나 거짓된 것을 믿게 하거나 위험한 행동을 하도록 설득하는 위험",
    },
    "능력이나 견고성의 부족": {
        "description": "AI 시스템이 설계된 목적을 달성하지 못하거나 예상치 못한 상황에서 예측 불가능하게 실패하는 기술적 한계",
    },
    "투명성, 해석가능성 부족": {
        "description": "AI 시스템의 의사결정 과정이나 작동 원리를 인간이 이해하거나 설명할 수 없는 블랙박스 상태",
    },
    "AI 복지 및 권리": {
        "description": "AI가 주관적인 경험, 특히 쾌락과 고통을 느낄 수 있는 능력을 획득하여, 지각이 있는데도 불구하고 지각이 없는 것으로 취급하는 경우 시스템이 학대받거나 해를 입는 위험",
    },
    "멀티 에이전트 위험": {
        "description": "여러 AI 시스템들이 동시에 상호작용할 때 발생하는 예측 불가능한 집단 행동이나 시스템 전체의 불안정성. 다중 AI 환경에서 발생하는 고유한 위험",
    },
}

def format_allowed_frames():
    """프레임을 프롬프트용 문자열로 변환"""
    frame_list = []
    for frame_name, frame_info in ALLOWED_FRAMES.items():
        frame_text = f"- {frame_name}: {frame_info['description']}"
        frame_list.append(frame_text)
    return "\n".join(frame_list)

# ═══════════════════════════════════════════════════════════════════════
# 3. 프롬프트
# ═══════════════════════════════════════════════════════════════════════
PROMPT_TEMPLATE = """다음 텍스트를 6개 AI RISK 프레임 중 하나로 분류해주세요.

[AI 관련 텍스트]: {text}

[AI RISK 프레임 목록]:
{allowed_frames}

위 6개 AI RISK 프레임 중 가장 적절한 하나를 선택하고, 그 이유를 설명하세요.
또한 이 분류에 대한 신뢰도를 1-100% 사이의 정수로 평가하세요.

**반드시 다음 JSON 형식으로만 답변하세요:**
{{
  "frame": "정확한 프레임 이름 (위 목록에서 선택)",
  "reason": "해당 프레임으로 분류한 근거와 이유",
  "confidence": "분류 신뢰도"
}}

답변 주의사항:
- frame, reason, confidence 모두 필수
- confidence는 1-100 사이의 정수
- 특수문자나 줄바꿈 사용 금지
- JSON 외 다른 내용 금지

JSON 응답:"""

# ═══════════════════════════════════════════════════════════════════════
# 4. 유틸리티 함수
# ═══════════════════════════════════════════════════════════════════════
def clean_text(text):
    """텍스트 정제"""
    if pd.isna(text) or text == "":
        return "본문 정보 없음"

    text = str(text)
    text = text.replace('\n', ' ')
    text = text.replace('\r', ' ')
    text = text.replace('\t', ' ')
    text = text.replace('\b', ' ')
    text = text.replace('\f', ' ')
    text = text.replace('\v', ' ')
    text = re.sub(r'[\x00-\x1f\x7f-\x9f]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text if text else "본문 정보 없음"

def parse_json(response):
    """JSON 파싱 (간소화)"""
    try:
        response = clean_text(response)
        response = re.sub(r'```json\s*', '', response)
        response = re.sub(r'```\s*', '', response)

        match = re.search(r'\{.*\}', response, re.DOTALL)
        if match:
            result = json.loads(match.group(0))

            frame = result.get("frame", "").strip()
            reason = result.get("reason", "").strip()
            confidence = result.get("confidence", 0)

            # Confidence 검증
            try:
                confidence = int(confidence)
                if not (1 <= confidence <= 100):
                    confidence = 50  # 기본값
            except:
                confidence = 50

            # 프레임 검증
            if frame not in ALLOWED_FRAMES:
                for allowed_frame in ALLOWED_FRAMES:
                    if allowed_frame.lower() in frame.lower() or frame.lower() in allowed_frame.lower():
                        frame = allowed_frame
                        break
                else:
                    frame = "파싱실패"
                    reason = f"허용되지 않은 프레임: {frame}"

            return frame, reason, confidence

        return "파싱실패", "JSON 형식 오류", 0

    except json.JSONDecodeError as e:
        logger.warning(f"JSON 파싱 실패: {e}")
        return "파싱실패", f"JSON 파싱 실패: {str(e)}", 0
    except Exception as e:
        logger.error(f"예외 발생: {e}")
        return "파싱실패", f"오류: {str(e)}", 0

# ═══════════════════════════════════════════════════════════════════════
# 5. 프레임 분석기 (EXAONE 4.0.1)
# ═══════════════════════════════════════════════════════════════════════
class AIRiskFrameAnalyzer:
    """EXAONE 4.0.1 기반 프레임 분석기"""

    def __init__(self, client, output_dir='./'):
        self.client = client
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)

        logger.info("🚀 EXAONE 4.0.1 프레임 분석기 초기화 완료!")
        logger.info("⚙️  설정: Non-reasoning 모드 + Greedy Decoding (temp=0.0)")

    def analyze_news(self, text):
        """단일 뉴스 프레임 분석"""
        allowed_frames_text = format_allowed_frames()
        prompt = PROMPT_TEMPLATE.format(text=text, allowed_frames=allowed_frames_text)

        try:
            response = self.client.chat.completions.create(
                model="lgai-exaone_exaone-4.0.1-32b",
                messages=[
                    {
                        "role": "system",
                        "content": "당신은 AI RISK를 정확히 분석하는 컴퓨터공학자입니다. 주어진 6개 프레임 중 텍스트에 해당하는 하나를 선택하여 JSON 형식으로 정확히 응답하세요."
                    },
                    {"role": "user", "content": prompt}
                ],
                temperature=0.0,
                max_tokens=8196,
                extra_body={
                    "chat_template_kwargs": {
                        "enable_thinking": False
                    }
                }
            )

            model_response = response.choices[0].message.content.strip()
            logger.debug(f"EXAONE 4.0.1 응답: {model_response[:200]}")

            frame, reason, confidence = parse_json(model_response)

            logger.info(f"✅ 프레임: {frame}, 신뢰도: {confidence}%")
            logger.info(f"🎯 이유: {reason[:50]}...")

            return frame, reason, confidence

        except Exception as e:
            logger.error(f"❌ EXAONE 4.0.1 분석 오류: {e}")
            return "파싱실패", f"분석 오류: {str(e)}", 0

    def process_dataframe(self, df, output_file, checkpoint_interval=10):
        """데이터프레임 배치 처리"""
        output_path = os.path.join(self.output_dir, output_file)
        checkpoint_path = os.path.join(self.output_dir, f"checkpoint_{output_file}")

        frame_names = []
        frame_reasons = []
        confidences = []

        # 체크포인트 복구
        start_idx = 0
        if os.path.exists(checkpoint_path):
            try:
                cp_df = pd.read_csv(checkpoint_path, encoding='utf-8-sig')
                if '프레임 이름' in cp_df.columns and '이유' in cp_df.columns:
                    completed = sum(~cp_df['프레임 이름'].isna())
                    if completed > 0:
                        start_idx = completed
                        frame_names = cp_df['프레임 이름'].tolist()[:start_idx]
                        frame_reasons = cp_df['이유'].tolist()[:start_idx]

                        # confidence 컬럼이 있으면 복구, 없으면 0으로 채움
                        if 'confidence' in cp_df.columns:
                            confidences = cp_df['confidence'].tolist()[:start_idx]
                        else:
                            confidences = [0] * start_idx

                        logger.info(f"📂 체크포인트 복구: {start_idx}행부터 재개")
            except Exception as e:
                logger.warning(f"⚠️  체크포인트 복구 실패: {e}")

        # 통계 변수
        success_count = len(frame_names)
        failure_count = 0

        # 진행 상황 표시
        pbar = tqdm(
            total=len(df) - start_idx,
            initial=0,
            desc="🚀 EXAONE 4.0.1 프레임 분석"
        )

        # 데이터 순차 처리
        for i in range(start_idx, len(df)):
            text = df.iloc[i]['text']

            if pd.isna(text) or text == "":
                text = "본문 정보 없음"

            logger.info(f"{'='*60}")
            logger.info(f"📰 기사 {i+1}/{len(df)} 처리 중")
            logger.info(f"{'='*60}")

            # 프레임 분석 (재시도 1회)
            try:
                frame, reason, confidence = self.analyze_news(text)

                if frame == "파싱실패":
                    logger.warning(f"⚠️  1차 시도 실패, 재시도 중...")
                    time.sleep(1)
                    frame, reason, confidence = self.analyze_news(text)

            except Exception as e:
                logger.error(f"❌ 행 {i} 처리 중 예외: {e}")
                frame = "파싱실패"
                reason = f"처리 오류: {str(e)}"
                confidence = 0

            # 결과 검증
            if frame == "파싱실패":
                failure_count += 1
                logger.warning(f"⚠️  처리 실패! (누적 실패: {failure_count}개)")
            else:
                success_count += 1
                logger.info(f"✅ 처리 성공! (누적 성공: {success_count}개)")

            # 결과 저장
            frame_names.append(frame)
            frame_reasons.append(reason)
            confidences.append(confidence)

            logger.info(f"📊 현재 진행률: {i+1}/{len(df)} (성공: {success_count}, 실패: {failure_count})")

            pbar.update(1)

            # 체크포인트 저장
            if (i + 1) % checkpoint_interval == 0 or i == len(df) - 1:
                temp_df = df.copy()
                temp_df['프레임 이름'] = frame_names + [''] * (len(temp_df) - len(frame_names))
                temp_df['이유'] = frame_reasons + [''] * (len(temp_df) - len(frame_reasons))
                temp_df['confidence'] = confidences + [0] * (len(temp_df) - len(confidences))
                temp_df.to_csv(checkpoint_path, encoding='utf-8-sig', index=False)
                logger.info(f"💾 체크포인트 저장: {len(frame_names)}/{len(df)}")

        pbar.close()

        # 최종 결과 생성
        result_df = df.copy()
        result_df['프레임 이름'] = frame_names
        result_df['이유'] = frame_reasons
        result_df['confidence'] = confidences

        # 최종 통계
        final_success_rate = (success_count / len(df)) * 100
        avg_confidence = sum(c for c in confidences if c > 0) / max(sum(1 for c in confidences if c > 0), 1)

        logger.info(f"\n{'='*60}")
        logger.info("🎉 EXAONE 4.0.1 프레임 분석 완료!")
        logger.info(f"{'='*60}")
        logger.info(f"📊 총 분석: {len(df)}개")
        logger.info(f"✅ 성공: {success_count}개")
        logger.info(f"❌ 실패: {failure_count}개")
        logger.info(f"📈 성공률: {final_success_rate:.2f}%")
        logger.info(f"🎯 평균 신뢰도: {avg_confidence:.1f}%")
        logger.info(f"💾 결과 저장: {output_path}")

        # 프레임 분포
        frame_distribution = result_df['프레임 이름'].value_counts()
        logger.info(f"\n📊 프레임 분포:\n{frame_distribution}")
        logger.info(f"{'='*60}\n")

        # 최종 저장
        result_df.to_csv(output_path, encoding='utf-8-sig', index=False)

        return result_df

# ═══════════════════════════════════════════════════════════════════════
# 6. 데이터 로드 함수
# ═══════════════════════════════════════════════════════════════════════
def load_and_clean_data(file_path):
    """데이터 로드 및 정제"""
    logger.info(f"📚 데이터 로드 중: {file_path}")
    df = pd.read_csv(file_path, encoding="utf-8-sig")

    logger.info("="*60)
    logger.info("📋 원본 데이터 정보")
    logger.info("="*60)
    logger.info(f"총 행 수: {len(df)}")
    logger.info(f"text 컬럼 NaN: {df['text'].isna().sum()}개")
    logger.info(f"text 컬럼 빈 문자열: {(df['text'] == '').sum()}개")

    df['text'] = df['text'].apply(clean_text)

    logger.info(f"✅ 데이터 정제 완료: {len(df)}개 기사")
    logger.info(f"⚠️  '본문 정보 없음' 처리: {(df['text'] == '본문 정보 없음').sum()}개")
    logger.info("="*60 + "\n")

    return df

# ═══════════════════════════════════════════════════════════════════════
# 7. 메인 함수
# ═══════════════════════════════════════════════════════════════════════
def main():
    # EXAONE 4.0.1 연결
    logger.info("="*60)
    logger.info("🔷 EXAONE 4.0.1 서버 연결 중...")
    logger.info("="*60)

    client = openai.OpenAI(
        base_url="http://218.156.182.231:1234/v1",
        api_key="not-needed"
    )

    # 연결 테스트
    try:
        test_response = client.chat.completions.create(
            model="lgai-exaone_exaone-4.0.1-32b",
            messages=[{"role": "user", "content": "안녕하세요"}],
            max_tokens=10
        )
        logger.info("✅ EXAONE 4.0.1 연결 성공!")
        logger.info("="*60 + "\n")
    except Exception as e:
        logger.error(f"❌ 연결 실패: {e}")
        exit(1)

    # 설정값 (사용자가 직접 입력)
    INPUT_FILE = "/mnt/c/Users/com/Desktop/test_4.0.1.csv"  # 여기에 입력 파일 경로 추가
    OUTPUT_FILE = "test_4.0.1_results.csv.csv"
    OUTPUT_DIR = "/mnt/c/Users/com/Desktop"  # 여기에 출력 디렉토리 경로 추가
    CHECKPOINT_INTERVAL = 10

    # 1. 데이터 로드
    df = load_and_clean_data(INPUT_FILE)

    # 2. 전체 분석
    logger.info("="*60)
    logger.info("🎯 EXAONE 4.0.1 AI Risk 프레임 분석 시작")
    logger.info("="*60)
    analyzer = AIRiskFrameAnalyzer(client, output_dir=OUTPUT_DIR)

    result_df = analyzer.process_dataframe(
        df,
        output_file=OUTPUT_FILE,
        checkpoint_interval=CHECKPOINT_INTERVAL
    )

    # 3. 완료
    logger.info("🎊 모든 작업 완료!")

if __name__ == "__main__":
    main()

In [ ]:
{
  "type": "object",
  "properties": {
    "frame": {
      "type": "string",
      "description": "프레임 이름"
    },
    "reason": {
      "type": "string",
      "description": "분류한 근거와 이유"
    },
    "confidence": {
      "type": "integer",
      "description": "분류 신뢰도 (1-100%)",
      "minimum": 1,
      "maximum": 100
    }
  },
  "required": [
    "frame",
    "reason",
    "confidence"
  ]
}